# permute-back-argsort — faded example 1: Call argsort to get the inverse permutation

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `permute-back-argsort`. Running the beacon reports progress on the `Backprop: permute_back via argsort` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: permute_back via argsort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`permute-back-argsort`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "permute-back-argsort"
DD_SUBTOPIC = "Backprop: permute_back via argsort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The backward pass of `permute` is `grad_out` permuted by the inverse of `dims`. The inverse permutation is computed by `np.argsort(dims)`, which returns the position of each integer `0..n-1` in `dims`. You then apply that inverse to `grad_out` with `grad_out.permute(*inverse)`.

## Faded exercise 1

### Exercise — Call argsort to get the inverse permutation

Complete `permute_back(grad_out, out, x, dims)`. The blank is the argsort call that computes the inverse permutation.

Fill in the line that computes `inverse` from `dims`.

**Fill in:** Compute the inverse permutation tuple using np.argsort on dims, converting to a Python int tuple.

In [ ]:
import torch as t
import numpy as np

def permute_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, dims: tuple) -> t.Tensor:
    inverse = None  # TODO: Compute the inverse permutation tuple using np.argsort on dims, converting to a Python int tuple.
    return grad_out.permute(*inverse)

t.manual_seed(3)
x = t.randn(3, 4, 5)
dims = (2, 0, 1)
y = x.permute(*dims)
grad_out = t.ones_like(y)
grad_x = permute_back(grad_out, y, x, dims)
print(grad_x.shape)  # should be (3, 4, 5)


def _test():
    import torch as t
    import numpy as np
    import itertools
    t.manual_seed(3)
    x = t.randn(3, 4, 5)
    for dims in itertools.permutations((0, 1, 2)):
        y = x.permute(*dims)
        grad_out = t.randn_like(y)
        grad_x = permute_back(grad_out, y, x, dims)
        assert grad_x.shape == x.shape, f'shape mismatch for dims={dims}'
        # Round-trip: permute(grad_out that equals y) should equal x
        assert t.equal(permute_back(y, y, x, dims), x), f'round-trip failed for dims={dims}'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import numpy as np

def permute_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, dims: tuple) -> t.Tensor:
    inverse = tuple(int(i) for i in np.argsort(dims))
    return grad_out.permute(*inverse)

t.manual_seed(3)
x = t.randn(3, 4, 5)
dims = (2, 0, 1)
y = x.permute(*dims)
grad_out = t.ones_like(y)
grad_x = permute_back(grad_out, y, x, dims)
print(grad_x.shape)  # should be (3, 4, 5)
```
</details>